# Variance

**Lesson:** Assumptions, Seasonality and Variance  
_Financial Forecasting in Python_

📄 Notes: [02 - Working with variances in the Forecast](docs/02%20-%20Working%20with%20variances%20in%20the%20Forecast.md)

## Overview

Measure and interpret the variance between forecasts and actuals.

## Learning objectives

- Define variance as the difference between forecast and actual figures, in absolute and percentage terms.
- Classify variances as favorable or unfavorable and interpret what they say about a forecast.
- Summarise forecast accuracy across periods with error metrics (MAE, RMSE, MAPE).

## Setup

In [8]:
import sys; sys.path.insert(0, '../../src')

# import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from ffp import error_report  # shared helper: MAE, RMSE, MAPE

## Walkthrough

In [9]:
# Variance is simply the gap between what we forecast and what actually happened.
# Positive variance = actual came in above forecast; negative = below.
forecast = 5000
actual = 5100

variance = actual - forecast
pct_variance = variance / forecast * 100

print("Forecast: {}".format(forecast))
print("Actual:   {}".format(actual))
print("Variance: {:+} ({:+.1f}%)".format(variance, pct_variance))

Forecast: 5000
Actual:   5100
Variance: +100 (+2.0%)


**Exercise 1 - Absolute variance per period**

A forecast is rarely spot on. **Variance** measures the gap between what was forecast and what actually happened, calculated for each period as `actual - forecast`.

Txs Tools forecast the following monthly sales, and later recorded the actual figures:

| Month | Forecast | Actual |
|-------|----------|--------|
| Jan   | 5000     | 5100   |
| Feb   | 5200     | 5000   |
| Mar   | 5400     | 5500   |
| Apr   | 5600     | 5400   |

Loop over the months and build a list `variances` containing the absolute variance (`actual - forecast`) for each month, printing the result per month.

In [10]:
# Forecast and actual monthly sales
months   = ['Jan', 'Feb', 'Mar', 'Apr']
forecast = [5000, 5200, 5400, 5600]
actual   = [5100, 5000, 5500, 5400]

# Build a list of the absolute variance for each month
variances = []
for i in range(len(months)):
    var = actual[i] - forecast[i]
    variances.append(var)
    print("{}: forecast={}, actual={}, variance={:+}".format(
        months[i], forecast[i], actual[i], var))

print("Variances: {}".format(variances))

Jan: forecast=5000, actual=5100, variance=+100
Feb: forecast=5200, actual=5000, variance=-200
Mar: forecast=5400, actual=5500, variance=+100
Apr: forecast=5600, actual=5400, variance=-200
Variances: [100, -200, 100, -200]


**Exercise 2 - Percentage variance, favorable or unfavorable**

An absolute variance of 100 means very different things on a 5,000 forecast versus a 500,000 one, so variance is usually also expressed as a percentage: `(actual - forecast) / forecast * 100`.

Variances are also labelled **favorable** or **unfavorable**. For a revenue line like sales, an actual figure **at or above** forecast is favorable; coming in below is unfavorable. (For a cost line the logic reverses.)

Using the same forecast and actual sales, print each month's **percentage variance** and whether it is favorable or unfavorable.

In [11]:
# Reuse the forecast and actual sales from Exercise 1
months   = ['Jan', 'Feb', 'Mar', 'Apr']
forecast = [5000, 5200, 5400, 5600]
actual   = [5100, 5000, 5500, 5400]

# For each month, report the percentage variance and whether it is favorable.
# For a revenue line, actual above forecast is favorable.
for i in range(len(months)):
    pct = (actual[i] - forecast[i]) / forecast[i] * 100
    status = "favorable" if actual[i] >= forecast[i] else "unfavorable"
    print("{}: {:+.1f}% ({})".format(months[i], pct, status))

Jan: +2.0% (favorable)
Feb: -3.8% (unfavorable)
Mar: +1.9% (favorable)
Apr: -3.6% (unfavorable)


**Exercise 3 - Summarising forecast accuracy**

Per-month variances are useful, but managers often want a single number for how accurate the forecast was overall. Three common summary metrics are:

- **MAE** (mean absolute error) — the average size of the variance, in currency units.
- **RMSE** (root mean squared error) — like MAE but penalises large misses more heavily.
- **MAPE** (mean absolute percentage error) — the average variance as a percentage of actuals.

The shared helper `error_report(y_true, y_pred)` returns all three. Using the same forecast and actual sales, print the MAE, RMSE and MAPE.

In [12]:
# Reuse the forecast and actual sales from Exercise 1
forecast = [5000, 5200, 5400, 5600]
actual   = [5100, 5000, 5500, 5400]

# error_report expects (y_true, y_pred) -> here actuals are the truth
report = error_report(actual, forecast)

print("MAE:  {:.2f}".format(report['mae']))    # average absolute variance
print("RMSE: {:.2f}".format(report['rmse']))   # penalises larger misses
print("MAPE: {:.2f}%".format(report['mape']))  # average variance as a % of actuals

MAE:  150.00
RMSE: 158.11
MAPE: 2.87%


**Exercise 4 - Gap analysis and an alternative forecast**

When actuals diverge from the forecast, we can no longer trust the original plan — we have to find the underlying cause and build an **alternative forecast**. Quantifying the difference between the old and revised forecast is called a **gap analysis**.

A company forecast sales of **1200** for 2018, based on selling **120 units** to a key supplier. Halfway through the year the gap analysis reveals the volume assumption was wrong: only **30 units** have been bought so far, and at most **45 more** are expected — a revised volume of just 75 units, not 120.

Work out the **unit price** implied by the original forecast, use it to build the **alternative forecast** from the revised volume, and report the **gap** against the original.

In [13]:
# Original forecast and the volume assumption behind it
original_forecast = 1200
original_volume = 120
unit_price = original_forecast / original_volume  # implied price per unit

# Revised volume uncovered by the gap analysis
units_bought = 30
units_expected = 45
revised_volume = units_bought + units_expected

# Build the alternative forecast and measure the gap
alternative_forecast = revised_volume * unit_price
gap = original_forecast - alternative_forecast

print("Implied unit price:   {:.2f}".format(unit_price))
print("Alternative forecast: {:.2f}".format(alternative_forecast))
print("Gap vs original:      {:.2f}".format(gap))

Implied unit price:   10.00
Alternative forecast: 750.00
Gap vs original:      450.00


In [14]:
import pandas as pd

# Netflix income statement (M $). Skip the "Actuals/Estimates" banner row so the
# fiscal-period row becomes the header, then name the label column.
netflix_f_is = pd.read_csv('../../data/netflix_forecast.csv', skiprows=1)
netflix_f_is = netflix_f_is.rename(columns={netflix_f_is.columns[0]: 'Description'})

print(netflix_f_is)

               Description    2014    2015    2016     2017     2018     2019
0                    Sales  5505.0  6780.0  8831.0  11688.0  14979.0  17994.0
1                   EBITDA   528.0   493.0   611.0   1088.0   1899.0   2943.0
2  Operating profit (EBIT)   403.0   306.0   380.0    837.0   1660.0   2702.0
3               Net income   267.0   123.0   187.0    559.0   1024.0   1721.0


**Exercise 5 - Building an alternative forecast**

We will now build an alternative forecast for Txs Tools. The new quarter forecast is based off actual data for Jul - Aug as well as adjusted forecast data for September. The data (units sold) is as follows:

* Jul = 700
* Aug = 220
* Sep = 520

The dependencies calculations have already been completed from the previous exercise. The following information applies:

* base_cost_price = 7
* base_sales_price = 15


In [15]:
def dependencies(base_cost_price, base_sales_price, sales_usd):
    res = []
    for sales in sales_usd:
        if sales >= 350:
            sales_dep = (350 * base_sales_price) + ((sales - 350) * (base_sales_price - 1))
        else:
            sales_dep = sales * base_sales_price
        if sales >= 500:
            cost_dep = (500 * base_cost_price) + ((sales - 500) * (base_cost_price + 2))
        else:
            cost_dep = sales * base_cost_price
        res.append(sales_dep - cost_dep)
    return res

forecast1 = dependencies(7, 15, [700, 350, 650])

forecast2 = dependencies(7, 15, [700, 220, 520])

print("The original forecast scenario is {}:".format(forecast1))
print("The alternative forecast scenario is {}:".format(forecast2))

The original forecast scenario is [4850, 2800, 4600]:
The alternative forecast scenario is [4850, 1760, 3950]:


**Exercise 6 - Building a gap analysis between forecasts**

Txs Tools now has two forecasts, the original forecast forecast1 and the adjusted forecast forecast2.

The dependencies have already been defined as def dependencies(base_cost_price, base_sales_price, sales_usd), where base_cost_price = 7 and base_sales_price = 15, with forecast2 based off the following adjusted sales unit values:

* Jul = 700
* Aug = 220
* Sep = 520


In this exercise, we will look at how to use a for loop to cycle between two different lists, forecast1 and forecast2 and calculate the difference ("gap") using an incremented index. It is possible to do this simultaneously as both lists have the same length.

In [16]:
# Set the two results
forecast1 = dependencies(7, 15, [700, 350, 650])
forecast2 = dependencies(7, 15, [700, 220, 520])

# Create an index and the gap analysis for the forecast
index = 0
for value in forecast2:
    print("The gap between forecasts is {}".format(value - forecast1[index]))
    index += 1

The gap between forecasts is 0
The gap between forecasts is -1040
The gap between forecasts is -650


**Exercise 7 - Setting dependencies for Netflix**

Netflix has compiled a forecast up to the 2019 financial year netflix_f_is, and has based the sales figures in 2019 on the following dependency:

- Number of active subscriptions, which are based on the success of Netflix original shows.

For 2019, the success of original shows (critical and commercial acclaim) are estimated at 78%. The total amount of subscribers per percentage point is 500, and set to the variable n_subscribers_per_pp (i.e there is a calculated correlation between show success and number of subscribers).

In this exercise, we will calculate how dependent sales are on the number of subscribers in the forecast, which we will use in the next exercise.

The Netflix forecast, netflix_f_iscan be printed in the shell.



In [ ]:
n_subscribers_per_pp = 500
show_success = 78  # % critical/commercial acclaim estimated for 2019

netflix_f_is = pd.read_csv('../../data/netflix_forecast.csv', skiprows=1)
netflix_f_is = netflix_f_is.rename(columns={netflix_f_is.columns[0]: 'Description'})
print(netflix_f_is)

# Subscribers implied by show success, and how much 2019 sales depend on each one
n_subscribers = show_success * n_subscribers_per_pp
sales_2019 = netflix_f_is.loc[0, '2019']  # Sales row (index 0), 2019 column
sales_per_subscriber = sales_2019 / n_subscribers

print("Number of subscribers: {}".format(n_subscribers))
print("Sales dependency per subscriber: {:.4f} M $".format(sales_per_subscriber))

## Summary

Variance is the gap between forecast and actual — measured per period in absolute
(`actual - forecast`) and percentage terms, labelled favorable or unfavorable, and
summarised across periods with MAE, RMSE and MAPE.

Recap the key points in the notes: [02 - Working with variances in the Forecast](docs/02%20-%20Working%20with%20variances%20in%20the%20Forecast.md)

Next up: [03 - Seasonality](03-seasonality.ipynb), which refines flat forecasts to reflect recurring seasonal patterns.